In [ ]:
#!/usr/bin/env python3
"""
Audit and diagnostic script for BANBAR across all ICICI Breeze API endpoints.
Queries:
1. Live Quotes & Scrip Details
2. Active MTF / Margin Positions (get_portfolio_positions)
3. Settled Demat Cash Holdings (get_portfolio_holdings)
4. Today's Order Book (Day / Limit / SL / Open / Executed / Cancelled)
5. GTT Triggers & Rule Orders
6. Historical Trade Book across Exchanges (NSE & BSE)
7. Customer Details & Fund Limits
"""

import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor
from breeze_core import get_breeze_client, FROM_DATE

TARGET_STOCK = "BANBAR"

def print_banner(title):
    print("\n" + "=" * 80)
    print(f"  >>> {title}")
    print("=" * 80)

def dump_records(label, records):
    print(f"\n--- {label} (Count: {len(records) if isinstance(records, list) else 1}) ---")
    if not records:
        print("[-] None found.")
        return
    print(json.dumps(records, indent=2, default=str))

def safe_filter(records, key="stock_code"):
    if not isinstance(records, list):
        return []
    return [
        r for r in records
        if str(r.get(key) or r.get("stock_name") or "").upper().strip() == TARGET_STOCK
    ]

def audit_banbar():
    breeze, is_auth = get_breeze_client()
    if not is_auth or not breeze:
        print("[!] Breeze API Authentication failed. Check your session/appkey/secret.")
        return

    print_banner(f"STARTING COMPREHENSIVE AUDIT FOR SCRIP: {TARGET_STOCK}")

    # -------------------------------------------------------------------------
    # 1. LIVE QUOTE & MARKET DATA
    # -------------------------------------------------------------------------
    print_banner("1. LIVE QUOTES (NSE)")
    try:
        quote_res = breeze.get_quotes(
            stock_code=TARGET_STOCK,
            exchange_code="NSE",
            product_type="cash"
        )
        quote = quote_res.get("Success", []) if isinstance(quote_res, dict) else []
        dump_records("NSE Cash Quote", quote)
    except Exception as e:
        print(f"[-] Quote error: {e}")

    # -------------------------------------------------------------------------
    # 2. OPEN MTF / MARGIN / DERIVATIVE POSITIONS
    # -------------------------------------------------------------------------
    print_banner("2. OPEN MTF / MARGIN POSITIONS (get_portfolio_positions)")
    try:
        pos_res = breeze.get_portfolio_positions()
        pos_list = pos_res.get("Success", []) if isinstance(pos_res, dict) else []
        banbar_mtf = safe_filter(pos_list, "stock_code")
        dump_records("Active MTF / Margin Lots", banbar_mtf)

        # Inspect MTF pledges and conversion flags
        total_mtf_qty = sum(float(p.get("quantity") or 0) for p in banbar_mtf)
        total_payable = sum(float(p.get("mtf_net_amount_payable") or 0) for p in banbar_mtf)
        print(f"\n[*] Active MTF Lots: {len(banbar_mtf)}")
        print(f"[*] Total Unsettled MTF Shares: {total_mtf_qty}")
        print(f"[*] Total Outstanding Margin Payable: ₹{total_payable:,.2f}")
    except Exception as e:
        print(f"[-] Positions error: {e}")

    # -------------------------------------------------------------------------
    # 3. DEMAT PORTFOLIO HOLDINGS (Settled Cash Delivery)
    # -------------------------------------------------------------------------
    print_banner("3. SETTLED DEMAT DELIVERY (get_portfolio_holdings)")
    try:
        hld_res = breeze.get_portfolio_holdings(exchange_code="NSE")
        hld_list = hld_res.get("Success", []) if isinstance(hld_res, dict) else []
        banbar_demat = safe_filter(hld_list, "stock_code")
        dump_records("Demat Delivery Holdings", banbar_demat)

        demat_qty = sum(float(h.get("quantity") or 0) for h in banbar_demat)
        print(f"\n[*] Total Demat Settled Shares: {demat_qty}")
    except Exception as e:
        print(f"[-] Portfolio Holdings error: {e}")

    # -------------------------------------------------------------------------
    # 4. ORDER BOOK (NSE & BSE)
    # -------------------------------------------------------------------------
    print_banner("4. ORDER BOOK (Today's Open / Executed / Cancelled Orders)")
    all_orders = []
    for ex in ["NSE", "BSE"]:
        try:
            ord_res = breeze.get_order_list(exchange_code=ex)
            items = ord_res.get("Success", []) if isinstance(ord_res, dict) else []
            filtered = safe_filter(items, "stock_code")
            for it in filtered:
                it["_exchange"] = ex
            all_orders.extend(filtered)
        except Exception as e:
            print(f"[-] Order list error ({ex}): {e}")
    dump_records("Today's Orders", all_orders)

    # -------------------------------------------------------------------------
    # 5. GTT & TRIGGER RULES
    # -------------------------------------------------------------------------
    print_banner("5. GTT & TRIGGER ORDERS")
    # Method A: Direct GTT endpoint if supported by Breeze client
    gtt_records = []
    try:
        if hasattr(breeze, "get_gtt_order"):
            gtt_res = breeze.get_gtt_order(exchange_code="NSE")
            items = gtt_res.get("Success", []) if isinstance(gtt_res, dict) else []
            gtt_records = safe_filter(items, "stock_code")
            dump_records("Direct GTT Order List", gtt_records)
    except Exception as e:
        print(f"[-] Direct GTT endpoint query error: {e}")

    # Method B: Scan standard order book and positions for stoploss triggers
    trigger_scans = []
    for o in all_orders:
        sl = float(o.get("stoploss") or o.get("trigger_price") or o.get("stoploss_trigger") or 0)
        otype = str(o.get("order_type") or "").lower()
        if sl > 0 or "sl" in otype or "gtt" in otype or "stoploss" in otype:
            trigger_scans.append(o)
    dump_records("Order Book Trigger / SL Rules", trigger_scans)

    # -------------------------------------------------------------------------
    # 6. HISTORICAL TRADE BOOK (Execution Fills)
    # -------------------------------------------------------------------------
    print_banner("6. TRADE BOOK (Historical Fills Since FROM_DATE)")
    to_date = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.000Z")
    from_date_iso = FROM_DATE if "T" in str(FROM_DATE) else f"{FROM_DATE}T06:00:00.000Z"

    all_trades = []
    for ex in ["NSE", "BSE"]:
        for ptype in ["", "cash", "margin"]:
            try:
                t_res = breeze.get_trade_list(
                    from_date=from_date_iso,
                    to_date=to_date,
                    exchange_code=ex,
                    product_type=ptype
                )
                items = t_res.get("Success", []) if isinstance(t_res, dict) else []
                filtered = safe_filter(items, "stock_code")
                for it in filtered:
                    it["_exchange"] = ex
                    it["_query_ptype"] = ptype or "ALL"
                    all_trades.append(it)
            except Exception as e:
                pass

    # Deduplicate trades by order_id / trade_date
    seen_trades = set()
    unique_trades = []
    for t in all_trades:
        key = (t.get("order_id"), t.get("trade_date"), t.get("quantity"), t.get("average_cost"))
        if key not in seen_trades:
            seen_trades.add(key)
            unique_trades.append(t)

    # Sort chronologically
    unique_trades.sort(key=lambda x: str(x.get("trade_date") or ""), reverse=False)
    dump_records("Complete Historical Trades", unique_trades)

    # -------------------------------------------------------------------------
    # 7. CONVERSION RECONCILIATION SUMMARY
    # -------------------------------------------------------------------------
    print_banner("7. BANBAR INVENTORY RECONCILIATION SUMMARY")
    total_bought_tradebook = 0
    total_mtf_bought = 0
    total_delivery_bought = 0

    for t in unique_trades:
        action = str(t.get("action") or "").upper()
        if "BUY" in action or action == "B":
            q = float(t.get("quantity") or 0)
            pt = str(t.get("product_type") or "").lower()
            total_bought_tradebook += q
            if "margin" in pt or "mtf" in pt:
                total_mtf_bought += q
            else:
                total_delivery_bought += q

    # Demat count
    demat_final = sum(float(h.get("quantity") or 0) for h in banbar_demat)
    mtf_open_final = sum(float(p.get("quantity") or 0) for p in banbar_mtf)

    # Converted MTF shares calculation
    converted_qty = max(0.0, total_mtf_bought - mtf_open_final)

    print(f"Trades Recorded in Trade Book:")
    print(f"  • Total Shares Purchased     : {int(total_bought_tradebook)}")
    print(f"  • MTF Purchases Logged       : {int(total_mtf_bought)}")
    print(f"  • Direct Delivery Buys Logged: {int(total_delivery_bought)}")
    print("-" * 50)
    print(f"Current Broker Ledger State:")
    print(f"  • Active Unexpired MTF Lots  : {int(mtf_open_final)}")
    print(f"  • Total Settled in Demat     : {int(demat_final)}")
    print("-" * 50)
    print(f"Conversion Analysis:")
    print(f"  • MTF Lots Converted to Demat: {int(converted_qty)} shares")
    print(f"  • Direct Demat + Converted   : {int(total_delivery_bought + converted_qty)} shares")

    if (total_delivery_bought + converted_qty) == demat_final:
        print("  ✓ MATH MATCHES 100%: All missing MTF shares sit accounted for inside Demat Delivery.")
    else:
        diff = demat_final - (total_delivery_bought + converted_qty)
        print(f"  ℹ Demat holds an additional {int(diff)} shares from transactions prior to {FROM_DATE}.")

if __name__ == "__main__":
    audit_banbar()

In [ ]:
import json
from datetime import datetime
from breeze_core import get_breeze_client, FROM_DATE

TARGET_STOCK = "BANBAR"

def inspect_stock(stock_code):
    breeze, is_auth = get_breeze_client()
    if not is_auth or not breeze:
        print("[-] Breeze authentication failed. Check your session token.")
        return

    to_date = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S.000Z")
    
    print(f"\n=======================================================")
    print(f"       RAW DATA INSPECTION FOR: {stock_code}")
    print(f"=======================================================\n")

    # 1. Demat Portfolio Holdings (Cash Depository)
    print(">>> 1. breeze.get_portfolio_holdings(exchange_code='NSE')")
    try:
        holdings = breeze.get_portfolio_holdings(exchange_code="NSE")
        filtered_holdings = []
        if holdings and isinstance(holdings.get("Success"), list):
            filtered_holdings = [
                h for h in holdings["Success"] 
                if str(h.get("stock_code") or "").strip().upper() == stock_code
            ]
        print(json.dumps(filtered_holdings if filtered_holdings else holdings, indent=2))
    except Exception as e:
        print(f"Error fetching holdings: {e}")

    # 2. Portfolio Positions (MTF / Margin / Day Positions)
    print("\n>>> 2. breeze.get_portfolio_positions()")
    try:
        positions = breeze.get_portfolio_positions()
        filtered_positions = []
        if positions and isinstance(positions.get("Success"), list):
            filtered_positions = [
                p for p in positions["Success"] 
                if str(p.get("stock_code") or "").strip().upper() == stock_code
            ]
        print(json.dumps(filtered_positions if filtered_positions else positions, indent=2))
    except Exception as e:
        print(f"Error fetching positions: {e}")

    # 3. Trade Book (NSE & BSE)
    print(f"\n>>> 3. breeze.get_trade_list(from_date='{FROM_DATE}', to_date='{to_date}')")
    for ex in ["NSE", "BSE"]:
        try:
            trades = breeze.get_trade_list(
                from_date=FROM_DATE,
                to_date=to_date,
                exchange_code=ex,
                product_type=""
            )
            filtered_trades = []
            if trades and isinstance(trades.get("Success"), list):
                filtered_trades = [
                    t for t in trades["Success"] 
                    if str(t.get("stock_code") or "").strip().upper() == stock_code
                ]
            print(f"--- Exchange: {ex} (Matched {len(filtered_trades)} records) ---")
            print(json.dumps(filtered_trades, indent=2))
        except Exception as e:
            print(f"Error fetching trades on {ex}: {e}")

    # 4. Order Book
    print("\n>>> 4. breeze.get_order_list(exchange_code='NSE')")
    try:
        orders = breeze.get_order_list(exchange_code="NSE")
        filtered_orders = []
        if orders and isinstance(orders.get("Success"), list):
            filtered_orders = [
                o for o in orders["Success"] 
                if str(o.get("stock_code") or "").strip().upper() == stock_code
            ]
        print(json.dumps(filtered_orders, indent=2))
    except Exception as e:
        print(f"Error fetching orders: {e}")

if __name__ == "__main__":
    inspect_stock(TARGET_STOCK)

In [ ]:
import csv
import time
from datetime import datetime, timedelta
import pandas as pd
from tqdm.notebook import tqdm
from breeze_core import get_breeze_client


def parse_date(date_str):
    if not date_str:
        return datetime.min
    for fmt in ("%d-%b-%Y", "%Y-%m-%d", "%d/%m/%Y", "%d-%b-%y"):
        try:
            return datetime.strptime(str(date_str).strip(), fmt)
        except ValueError:
            pass
    return datetime.min


def run_pipeline():
    print("[AUTH] Connecting to ICICI Breeze API...")
    breeze, is_auth = get_breeze_client()
    if not is_auth or not breeze:
        print("[ERROR] Authentication failed.")
        return
    print("[SUCCESS] Authenticated.\n")

    # 1. Fetch Active MTF Positions
    print("[API CALL] get_portfolio_positions()...")
    mtf_positions = []
    try:
        pos_res = breeze.get_portfolio_positions()
        mtf_positions = pos_res.get("Success", []) or []
        print(f"[API RESPONSE] Active MTF Positions: {len(mtf_positions)}")
    except Exception as e:
        print(f"[API ERROR] get_portfolio_positions: {e}")

    # 2. Fetch Settled Demat Holdings
    print("\n[API CALL] get_portfolio_holdings()...")
    demat_holdings = []
    for ex in ["NSE", "BSE"]:
        try:
            hold_res = breeze.get_portfolio_holdings(exchange_code=ex)
            hold_items = hold_res.get("Success", []) or []
            print(f"[API RESPONSE] Demat holdings ({ex}): {len(hold_items)}")
            if isinstance(hold_items, list):
                demat_holdings.extend(hold_items)
        except Exception as e:
            print(f"[API ERROR] get_portfolio_holdings ({ex}): {e}")

    # 3. Fetch Historical Trade Book (20-day windows to avoid 600 cap)
    start_date = datetime(2026, 1, 1)
    end_date = datetime.utcnow()
    chunk_days = 20

    date_windows = []
    curr = start_date
    while curr < end_date:
        nxt = min(curr + timedelta(days=chunk_days), end_date)
        date_windows.append((
            curr.strftime("%Y-%m-%dT00:00:00.000Z"),
            nxt.strftime("%Y-%m-%dT23:59:59.000Z"),
            curr.strftime("%Y-%m-%d"),
            nxt.strftime("%Y-%m-%d")
        ))
        curr = nxt + timedelta(seconds=1)

    raw_trades = []
    seen_trades = set()

    print(f"\n[API CALL] Fetching trades across {len(date_windows)} windows...")
    pbar = tqdm(date_windows, desc="Trade Windows", unit="window")

    for f_iso, t_iso, f_lbl, t_lbl in pbar:
        pbar.set_postfix_str(f"{f_lbl} to {t_lbl}")
        for ex in ["NSE", "BSE"]:
            try:
                res = breeze.get_trade_list(
                    from_date=f_iso,
                    to_date=t_iso,
                    exchange_code=ex,
                    product_type=""  # Returns both Cash and Margin
                )
                items = res.get("Success", []) or []
                count = len(items) if isinstance(items, list) else 0

                if count > 0:
                    tqdm.write(f"[API HIT] {ex} ({f_lbl} -> {t_lbl}): {count} trades")

                if isinstance(items, list):
                    for item in items:
                        uid = (
                            item.get("order_id"),
                            item.get("trade_date"),
                            item.get("stock_code"),
                            item.get("action"),
                            item.get("quantity"),
                            item.get("average_cost")
                        )
                        if uid not in seen_trades:
                            seen_trades.add(uid)
                            raw_trades.append(item)
            except Exception as e:
                tqdm.write(f"[API ERROR] get_trade_list ({ex} | {f_lbl}): {e}")

            time.sleep(0.08)

    raw_trades.sort(key=lambda x: parse_date(x.get("trade_date")))
    print(f"\n[SUMMARY] Total unique trades: {len(raw_trades)}")

    # 4. Export Raw Trade, MTF, and Demat Files
    def export_csv(records, filename):
        if not records:
            return
        keys = list({k for r in records for k in r.keys()})
        with open(filename, mode="w", newline="", encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            for r in records:
                writer.writerow(r)
        print(f"[SAVED] {len(records)} rows written to '{filename}'")

    print("\n--- Exporting Datasets ---")
    export_csv(raw_trades, "breeze_all_trades_2026.csv")
    export_csv(mtf_positions, "breeze_mtf_positions.csv")
    export_csv(demat_holdings, "breeze_demat_holdings.csv")

    # 5. Reconcile MTF-to-Delivery Conversions
    print("\n" + "=" * 80)
    print("MTF -> DELIVERY RECONCILIATION SUMMARY")
    print("=" * 80)

    mtf_buys = {}
    mtf_sells = {}
    for t in raw_trades:
        pt = str(t.get("product_type", "")).strip().lower()
        if pt in ["margin", "easymargin", "mtf"]:
            scrip = t.get("stock_code")
            qty = float(t.get("quantity") or 0)
            act = str(t.get("action", "")).strip().lower()
            if act == "buy":
                mtf_buys[scrip] = mtf_buys.get(scrip, 0) + qty
            elif act == "sell":
                mtf_sells[scrip] = mtf_sells.get(scrip, 0) + qty

    current_mtf = {p.get("stock_code"): float(p.get("quantity") or 0) for p in mtf_positions}
    current_demat = {h.get("stock_code"): float(h.get("quantity") or 0) for h in demat_holdings}

    conversions = []
    for scrip in sorted(mtf_buys.keys()):
        tot_buy = mtf_buys.get(scrip, 0)
        tot_sell = mtf_sells.get(scrip, 0)
        open_mtf = current_mtf.get(scrip, 0)
        demat = current_demat.get(scrip, 0)

        expected_open = tot_buy - tot_sell
        converted_qty = max(0.0, expected_open - open_mtf)

        if converted_qty > 0 and demat > 0:
            rec = {
                "stock_code": scrip,
                "mtf_bought": tot_buy,
                "mtf_sold": tot_sell,
                "remaining_open_mtf": open_mtf,
                "demat_holding": demat,
                "converted_to_delivery_qty": min(converted_qty, demat)
            }
            conversions.append(rec)
            print(f"Scrip: {scrip:<10} | Inferred Converted: {rec['converted_to_delivery_qty']} shares | Open MTF: {open_mtf} | Demat: {demat}")

    if conversions:
        export_csv(conversions, "breeze_conversions_inferred.csv")
    else:
        print("No MTF-to-Delivery conversions detected among the reconciled positions.")


run_pipeline()